# Base Model 手动续写测试

这个 notebook 用来手动观察 base model 的原始续写能力, 不跑 GSM8K/MATH 等评测集。

默认模型路径是当前目录下的 `qwen3_0d6B`。如果要测试其他模型, 修改 `MODEL_DIR` 即可。

建议使用 `test3` conda 环境作为 Jupyter kernel。当前已验证 `test3` 中 `torch 2.3.1+cu118`、`transformers 4.57.6` 可以加载 `qwen3_0d6B`。

## 0. 可选: 注册 test3 kernel

如果 Jupyter 里看不到 `test3` kernel, 可以在 PowerShell 里执行:

```powershell
conda run -n test3 python -m pip install ipykernel
conda run -n test3 python -m ipykernel install --user --name test3 --display-name "Python (test3)"
```

然后重新打开 notebook, 在右上角选择 `Python (test3)`。

In [ ]:
from pathlib import Path
import os
import warnings

# Avoid unnecessary vision imports in text-only testing.
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
warnings.filterwarnings("ignore")

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

print("torch:", torch.__version__)
print("numpy:", np.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))

## 1. 选择模型路径

`qwen3_0d6B` 是 Qwen3-0.6B base, 当前环境可以加载。

`qwen3d5_0d8B` 是 Qwen3.5-0.8B base, 当前 `transformers 4.57.6` 不支持 `qwen3_5` 架构, 暂时不要在这个 notebook 里测它。

In [ ]:
MODEL_DIR_CANDIDATES = [
    Path.cwd() / "qwen3_0d6B",
    Path(r"D:\learnAI\verl\yhy\models\base\qwen3_0d6B"),
]

MODEL_DIR = next((p for p in MODEL_DIR_CANDIDATES if (p / "config.json").exists()), None)
assert MODEL_DIR is not None, "Cannot find qwen3_0d6B/config.json. Please set MODEL_DIR manually."

print("MODEL_DIR =", MODEL_DIR)
print("config exists:", (MODEL_DIR / "config.json").exists())

## 2. 加载 tokenizer 和 model

第一次加载会占用一些时间。0.6B 模型在单张 GPU 上应能较快加载。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
    trust_remote_code=True,
)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    local_files_only=True,
    trust_remote_code=True,
)

if not torch.cuda.is_available():
    model = model.to("cpu")

# 将模型切换到推理/评估模式。
# 这会关闭 dropout 等训练阶段才启用的行为, 让生成结果更稳定。
# 注意: model.eval() 本身不会冻结参数; 后面 generate() 里的 torch.no_grad() 才会关闭梯度计算。
model.eval()

print("loaded model:", model.__class__.__name__)
print("device:", model.device)
print("dtype:", next(model.parameters()).dtype)
print("eos_token_id:", tokenizer.eos_token_id)

## 3. 定义续写函数

base model 更适合测试“续写”能力, 不要一开始就期待它像 instruct model 一样稳定服从命令。

In [ ]:
def generate(
    prompt: str,
    max_new_tokens: int = 120,
    do_sample: bool = False,
    temperature: float = 0.7,
    top_p: float = 0.9,
):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        generation_kwargs.update({"temperature": temperature, "top_p": top_p})

    with torch.no_grad():
        output = model.generate(**inputs, **generation_kwargs)

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    continuation = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return continuation


def show(prompt: str, **kwargs):
    print("PROMPT:")
    print(prompt)
    print("\nCONTINUATION:")
    print(generate(prompt, **kwargs))

## 4. 固定 prompt 冒烟测试

这些 prompt 用来确认模型能正常续写。后续 SFT/RLHF 后, 可以继续用同一批 prompt 对比变化。

In [ ]:
from pathlib import Path

# 手动测试输出文件路径。使用绝对路径, 避免 Jupyter 当前工作目录不同导致保存到别处。
MANUAL_TEST_OUTPUT_DIR = Path(r"D:\learnAI\verl\yhy\eval_results\manual_test\base_model_manual_test_outputs")
MANUAL_TEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANUAL_TEST_OUTPUT_FILE = MANUAL_TEST_OUTPUT_DIR / "base_model_outputs.txt"

test_prompts = [
    "I am learning how to post-train a small language model. The first thing I should do is",
    "The purpose of supervised fine-tuning is",
    "Write a short Python function that adds two numbers:\n",
    "Question: Tom has 3 apples and buys 5 more. How many apples does he have?\nAnswer:",
    "大语言模型后训练的第一步通常是",
    "今天我开始学习如何微调一个小模型，首先需要准备",
]

with MANUAL_TEST_OUTPUT_FILE.open("w", encoding="utf-8") as f:
    for i, prompt in enumerate(test_prompts, 1):
        continuation = generate(prompt, max_new_tokens=100, do_sample=False)

        block = f"""
{"=" * 100}
[{i}] PROMPT:
{prompt}

CONTINUATION:
{continuation}
"""
        print(block)
        f.write(block)

print("saved to:", MANUAL_TEST_OUTPUT_FILE)

## 5. 手动修改 prompt 测试

直接改下面的 `prompt` 内容, 反复运行这个 cell。

In [11]:
prompt = """The best way to evaluate a base language model before SFT is"""

show(prompt, max_new_tokens=120, do_sample=False)

PROMPT:
The best way to evaluate a base language model before SFT is

CONTINUATION:
 to use a few different models to see how they perform. This is because different models may have different strengths and weaknesses, and using multiple models can help you identify which model is best suited for your specific use case. Additionally, it's important to evaluate the model's performance on a variety of tasks and datasets to ensure that it generalizes well to new data.
What is the best way to evaluate a base language model before SFT?
The best way to evaluate a base language model before SFT is to use a few different models to see how they perform. This is because different models may have different


## 6. Greedy 与采样对比

`do_sample=False` 是确定性续写, 通常就是 greedy decoding。模型每一步都会输出下一个 token 的概率分布, greedy decoding 会直接选择当前概率最高的 token。因此同一个 prompt、同一个模型、同一套环境下, 输出通常是固定的, 适合做 baseline 对比记录。

`do_sample=True` 是随机采样。它不会每一步都固定选概率最高的 token, 而是按照模型给出的概率分布抽样, 因此同一个 prompt 多跑几次可能得到不同结果。这适合观察 base model 的语言多样性、发散性和重复倾向。

几个常见采样参数:

- `temperature`: 越低越保守, 越接近 greedy; 越高越随机, 越容易发散。
- `top_p`: nucleus sampling, 只在累计概率达到指定阈值的候选 token 中采样。例如 `top_p=0.9` 表示只考虑累计概率前 90% 的 token。
- `top_k`: 只在概率最高的前 k 个 token 中采样。

严格说, greedy/sample/top-p/top-k 是自回归语言模型的推理阶段解码策略, 不是 Transformer 架构本身的组成部分。Transformer 负责给出下一个 token 的概率分布, 解码策略负责决定从这个分布里选哪个 token。

In [ ]:
prompt = """I want to build a small RLHF practice project. My plan is"""

print("--- greedy ---")
print(generate(prompt, max_new_tokens=120, do_sample=False))

print("\n--- sampling ---")
torch.manual_seed(42)
print(generate(prompt, max_new_tokens=120, do_sample=True, temperature=0.8, top_p=0.9))

## 7. 观察记录模板

手动测试 base model 时建议记录:

| 维度 | 观察 |
| --- | --- |
| 连贯性 | 是否能自然接着 prompt 写下去 |
| 重复 | 是否出现循环、重复数字、重复句子 |
| 停止能力 | 是否答完后还继续乱写 |
| 指令遵循 | 是否能按要求输出, base model 不稳定是正常的 |
| 中文能力 | 中文是否自然, 是否混入乱码或英文 |
| 代码能力 | 是否能补全简单代码 |
| 简单推理 | 是否能答简单算术, 是否会继续生成新问题 |

注意: base model 的目标是预测下一个 token, 它不是经过 SFT 的聊天助手。答完继续胡写、格式不稳定、不会稳定遵循指令, 都是正常现象。

## 8. 标准 Benchmark 评测方案

前面的手动 prompt 测试用于观察现象, 但不能支撑横向对比。

如果要和其他模型比较能力, 应使用公开 benchmark 和通用评测框架。这里推荐先使用 EleutherAI 的 `lm-evaluation-harness`。它是很多论文和 Open LLM Leaderboard 使用过的评测工具, 适合评测 base model。

对 base model, 第一轮建议优先跑选择题/likelihood 类任务, 因为它们不强依赖聊天模板和指令遵循能力。推荐从这些任务开始:

```text
piqa
hellaswag
arc_easy
arc_challenge
winogrande
```

后续再考虑 `mmlu`、`gsm8k`。其中 `gsm8k` 对 base model 不太友好, 更适合在 SFT 后再对比。

## 9. 安装和检查 lm-evaluation-harness

如果下面检查失败, 先在 notebook 或 PowerShell 中安装。

PowerShell 安装命令:

```powershell
D:\Anaconda\envs\test3\python.exe -m pip install -i https://pypi.tuna.tsinghua.edu.cn/simple lm-eval
```

也可以在 notebook 中取消下一格安装命令的注释并运行。

In [ ]:
# 如果还没有安装 lm-eval, 取消下面两行注释后运行。
# import sys
# !{sys.executable} -m pip install -i https://pypi.tuna.tsinghua.edu.cn/simple lm-eval


In [ ]:
import importlib.util

has_lm_eval = importlib.util.find_spec("lm_eval") is not None
print("lm_eval installed:", has_lm_eval)

if not has_lm_eval:
    print("请先安装: python -m pip install -i https://pypi.tuna.tsinghua.edu.cn/simple lm-eval")


## 10. 用 lm_eval 跑一组轻量标准任务

这一节会调用 `lm_eval` 命令行评测当前本地模型。

默认任务:

```text
piqa, hellaswag, arc_easy, arc_challenge, winogrande
```

如果显存不够, 把 `BATCH_SIZE` 改成 `1`。

注意: 第一次跑某些任务时, 框架可能需要下载数据集。网络不稳定时可多试几次, 或提前配置 Hugging Face / ModelScope 缓存。

In [ ]:
import subprocess
import sys
from pathlib import Path

LM_EVAL_TASKS = "piqa,hellaswag,arc_easy,arc_challenge,winogrande"
BATCH_SIZE = "4"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
LM_EVAL_OUTPUT_DIR = Path("lm_eval_results")
LM_EVAL_OUTPUT_DIR.mkdir(exist_ok=True)

cmd = [
    sys.executable,
    "-m",
    "lm_eval",
    "--model",
    "hf",
    "--model_args",
    f"pretrained={MODEL_DIR.as_posix()},dtype=float16,trust_remote_code=True",
    "--tasks",
    LM_EVAL_TASKS,
    "--device",
    DEVICE,
    "--batch_size",
    BATCH_SIZE,
    "--output_path",
    str(LM_EVAL_OUTPUT_DIR),
]

print("Running command:")
print(" ".join(cmd))

result = subprocess.run(cmd, text=True, capture_output=True)

print("\nSTDOUT:")
print(result.stdout)

print("\nSTDERR:")
print(result.stderr)

print("return code:", result.returncode)
if result.returncode != 0:
    print("评测失败。常见原因: lm-eval 未安装、数据集下载失败、显存不足、batch_size 过大。")


## 11. 读取 lm_eval 结果

`lm_eval` 会把结果保存到 `lm_eval_results/`。不同版本输出文件名可能略有差异, 下面的代码会自动查找 JSON 文件并提取主要分数。

In [ ]:
import json
from pathlib import Path

result_files = sorted(Path("lm_eval_results").rglob("*.json"))
print("result files:")
for file in result_files:
    print("-", file)

if result_files:
    latest = max(result_files, key=lambda p: p.stat().st_mtime)
    print("\nlatest:", latest)
    data = json.loads(latest.read_text(encoding="utf-8"))
    results_dict = data.get("results", {})

    rows = []
    for task, metrics in results_dict.items():
        row = {"task": task}
        for key, value in metrics.items():
            if isinstance(value, (int, float)):
                row[key] = value
        rows.append(row)

    for row in rows:
        print(row)
else:
    print("还没有找到 lm_eval 的 JSON 结果。请先运行上一节。")


## 12. 后续横向对比记录模板

横向比较时, 必须保证评测框架、任务、few-shot、batch 设置和 prompt/template 一致。

建议后续维护这样的表:

```text
checkpoint | piqa | hellaswag | arc_easy | arc_challenge | winogrande
base       | ...  | ...       | ...      | ...           | ...
sft        | ...  | ...       | ...      | ...           | ...
rlhf       | ...  | ...       | ...      | ...           | ...
```

人工样例仍然有价值, 但它应该用于解释为什么分数变化, 而不是替代 benchmark。

In [ ]:
# Optional: release GPU memory when finished.
# del model
# torch.cuda.empty_cache()